# 04 — IPCA e deflacionamento de V24

**Objetivo:** deflacionar a variável de renda (V24) de 2000 e 2010 para reais de julho de 2022 — a data de
referência do Censo 2022 — usando o índice IPCA do IBGE (tabela SIDRA 1737, variável 2266).

**Entradas:**
- `output/03_compatibilizado/<slug_cidade>_contagens.gpkg` (5 arquivos)
- API SIDRA (`https://apisidra.ibge.gov.br`), com *cache* local em `output/04_deflacionado/ipca_indice.csv`

**Saídas:**
- `output/04_deflacionado/<slug_cidade>_deflacionado.gpkg` (5 arquivos)
- `output/04_deflacionado/ipca_indice.csv` (cache do índice, para rodar offline depois da primeira vez)

**Depende de:** `03_compatibilizacao_areal.ipynb`

**Premissa do projeto:** deflacionar para julho/2022 (mês de referência do Censo 2022), de modo que os valores
de 2022 permaneçam nominais (fator 1) e sirvam de base de comparação para 2000 e 2010.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "utils.py").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))

import geopandas as gpd
import pandas as pd
import requests

from utils import CIDADES, OUTPUT_DIR, VAR_RENDA

IN_DIR = OUTPUT_DIR / "03_compatibilizado"
OUT_DIR = OUTPUT_DIR / "04_deflacionado"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_IPCA = OUT_DIR / "ipca_indice.csv"

# Tabela SIDRA 1737, variável 2266: IPCA - Número-índice (base: dezembro de 1993 = 100)
MESES_REFERENCIA = {2000: "200007", 2010: "201007", 2022: "202207"}  # julho de cada ano do censo
ANO_BASE_DEFLACAO = 2022


## 2. Obtenção do índice IPCA (com cache local)

Consultamos a API do SIDRA para os três meses de referência. Se a consulta falhar (sem internet, API fora do ar),
usamos o cache gravado na execução anterior — e paramos com um erro claro se não houver nem API nem cache, em vez
de seguir com um valor arbitrário.

In [ ]:
def busca_ipca_sidra(meses_codigo):
    codigos = ",".join(meses_codigo.values())
    url = f"https://apisidra.ibge.gov.br/values/t/1737/n1/all/v/2266/p/{codigos}"
    resposta = requests.get(url, timeout=30)
    resposta.raise_for_status()
    dados = resposta.json()[1:]  # primeira linha é o cabeçalho de metadados

    linhas = []
    for ano, codigo in meses_codigo.items():
        registro = next(d for d in dados if d["D3C"] == codigo)
        linhas.append({"ano_censo": ano, "mes_referencia": codigo, "mes_nome": registro["D3N"],
                        "indice_ipca": float(registro["V"])})
    return pd.DataFrame(linhas)


try:
    df_ipca = busca_ipca_sidra(MESES_REFERENCIA)
    df_ipca.to_csv(CACHE_IPCA, index=False, encoding="utf-8-sig")
    print("Índice IPCA obtido da API SIDRA e salvo em cache.")
except Exception as erro:
    if CACHE_IPCA.exists():
        print(f"Falha ao consultar a API SIDRA ({erro}); usando cache local em {CACHE_IPCA}.")
        df_ipca = pd.read_csv(CACHE_IPCA)
    else:
        raise RuntimeError(
            "Não foi possível obter o índice IPCA (API indisponível) e não há cache local. "
            "Preencha manualmente output/04_deflacionado/ipca_indice.csv com colunas "
            "ano_censo, mes_referencia, mes_nome, indice_ipca antes de continuar."
        ) from erro

df_ipca


## 3. Fatores de deflação

`fator_ano = índice(jul/2022) / índice(jul/ano)`. Por construção, o fator de 2022 é 1 (valores nominais = reais).

In [ ]:
indice_base = df_ipca.loc[df_ipca["ano_censo"] == ANO_BASE_DEFLACAO, "indice_ipca"].iloc[0]
df_ipca["fator_deflacao_para_2022"] = indice_base / df_ipca["indice_ipca"]
df_ipca


**Checagem de plausibilidade:** o fator de 2000→2022 deve ficar perto de ~3,5-4,5× (a inflação acumulada em 22
anos no Brasil é grande, mas não descontrolada) e o de 2010→2022 perto de ~1,8-2,3×. Um valor muito fora dessa
faixa indica mês ou tabela SIDRA errados — não deflacionar sem confirmar.

In [ ]:
fator_2000 = df_ipca.loc[df_ipca["ano_censo"] == 2000, "fator_deflacao_para_2022"].iloc[0]
fator_2010 = df_ipca.loc[df_ipca["ano_censo"] == 2010, "fator_deflacao_para_2022"].iloc[0]

assert 3.0 < fator_2000 < 5.0, f"Fator 2000->2022 fora do esperado: {fator_2000:.3f}"
assert 1.5 < fator_2010 < 2.5, f"Fator 2010->2022 fora do esperado: {fator_2010:.3f}"

print(f"Fator 2000 -> 2022: {fator_2000:.4f}")
print(f"Fator 2010 -> 2022: {fator_2010:.4f}")
print("Fatores dentro da faixa de plausibilidade.")


## 4. Aplicação: colunas de renda deflacionada

- Mantém as colunas nominais originais (`V24_2000`, `V24_2010`) e cria as colunas deflacionadas
  (`V24_2000_r2022`, `V24_2010_r2022`), para auditar as duas versões lado a lado.
- `V24_2022` não muda (fator 1).

In [ ]:
fatores = df_ipca.set_index("ano_censo")["fator_deflacao_para_2022"].to_dict()

for cidade in CIDADES:
    slug = CIDADES[cidade]["slug"]
    gdf = gpd.read_file(IN_DIR / f"{slug}_contagens.gpkg")

    for ano in (2000, 2010):
        col_nominal = f"{VAR_RENDA}_{ano}"
        col_real = f"{VAR_RENDA}_{ano}_r2022"
        gdf[col_real] = gdf[col_nominal] * fatores[ano]

    gdf.to_file(OUT_DIR / f"{slug}_deflacionado.gpkg", driver="GPKG")
    print(f"{cidade}: colunas de renda deflacionada adicionadas -> {slug}_deflacionado.gpkg")


## 5. Conferência

In [ ]:
slug_exemplo = CIDADES["Anápolis"]["slug"]
gdf_exemplo = gpd.read_file(OUT_DIR / f"{slug_exemplo}_deflacionado.gpkg")
gdf_exemplo[["CD_SETOR", "V24_2000", "V24_2000_r2022", "V24_2010", "V24_2010_r2022", "V24_2022"]].describe()
